## Phase 6 — Change Data Feed (CDF) → Delta Analytics
**Source:** Delta CDF on `main.silver.*` tables
**Writes to:** `main.analytics.cdf_events`

Captures every INSERT/UPDATE/DELETE across the Silver pipeline tables
and streams them into a Delta analytics table for usage tracking.

Note: This uses Delta Lake native CDF (same concept as Lakebase CDF).
Lakebase CDF is architecturally identical — both use REPLICA IDENTITY FULL
(Postgres) / delta.enableChangeDataFeed (Delta) to capture row-level changes.
Direct Lakebase connectivity requires OAuth federation not available on Free Edition.


In [ ]:
# 0. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

PROCESSED_AT = datetime.now().isoformat()
print(f"CDF pipeline started: {PROCESSED_AT}")


In [ ]:
# 1. Enable CDF on all Silver tables
# This mirrors what REPLICA IDENTITY FULL does in Lakebase Postgres
print("\n--- Enabling CDF on Silver tables ---")

silver_tables = [
    "main.silver.companies",
    "main.silver.price_snapshots",
    "main.silver.news_articles",
    "main.silver.news_for_search"
]

for table in silver_tables:
    try:
        spark.sql(f"""
            ALTER TABLE {table}
            SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"  CDF enabled → {table} ✓")
    except Exception as e:
        print(f"  {table}: {e}")


In [ ]:
# 2. Create analytics schema and CDF events table
print("\n--- Creating analytics schema + cdf_events table ---")

spark.sql("CREATE SCHEMA IF NOT EXISTS main.analytics")

spark.sql("""
    CREATE TABLE IF NOT EXISTS main.analytics.cdf_events (
        event_id        STRING,
        source_table    STRING,
        operation       STRING,
        ticker          STRING,
        record_key      STRING,
        record_snapshot STRING,
        commit_version  BIGINT,
        commit_ts       TIMESTAMP,
        captured_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Row-level change events captured from Silver Delta tables via CDF'
""")

print("Schema main.analytics ready ✓")
print("Table main.analytics.cdf_events ready ✓")


In [ ]:
# 3. Read CDF from Silver tables and write to analytics
# Uses Delta CDF batch read — reads all changes since version 0
print("\n--- Reading CDF from Silver tables ---")

import uuid

def read_cdf_and_append(source_table: str, key_col: str, snapshot_cols: list):
    """
    Reads CDF from a Silver Delta table starting from version 0.
    Extracts operation type (INSERT/UPDATE/DELETE) and row snapshot.
    Appends to main.analytics.cdf_events.
    """
    print(f"\n  Reading CDF from {source_table}...")
    try:
        # Read CDF — includes _change_type, _commit_version, _commit_timestamp
        cdf_df = (
            spark.read
            .format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", 0)
            .table(source_table)
        )

        # Filter out update_preimage (keep only the new values)
        cdf_df = cdf_df.filter(
            F.col("_change_type").isin(
                "insert", "update_postimage", "delete"
            )
        )

        if cdf_df.count() == 0:
            print(f"  No changes found in {source_table}")
            return 0

        # Build a snapshot string from key columns
        snapshot_expr = F.to_json(
            F.struct(*[F.col(c) for c in snapshot_cols if c in cdf_df.columns])
        )

        # Map change types to readable operations
        events = (
            cdf_df
            .withColumn("event_id",
                F.concat(F.lit(str(uuid.uuid4())[:8] + "-"), F.monotonically_increasing_id().cast("string"))
            )
            .withColumn("source_table",   F.lit(source_table))
            .withColumn("operation",
                F.when(F.col("_change_type") == "insert",           "INSERT")
                 .when(F.col("_change_type") == "update_postimage", "UPDATE")
                 .when(F.col("_change_type") == "delete",           "DELETE")
                 .otherwise(F.col("_change_type"))
            )
            .withColumn("ticker",
                F.col("ticker") if "ticker" in cdf_df.columns else F.lit(None)
            )
            .withColumn("record_key",      F.col(key_col).cast("string"))
            .withColumn("record_snapshot", snapshot_expr)
            .withColumn("commit_version",  F.col("_commit_version"))
            .withColumn("commit_ts",       F.col("_commit_timestamp"))
            .withColumn("captured_at",     F.lit(PROCESSED_AT).cast("timestamp"))
            .select(
                "event_id", "source_table", "operation",
                "ticker", "record_key", "record_snapshot",
                "commit_version", "commit_ts", "captured_at"
            )
        )

        count = events.count()
        (events
         .write.format("delta")
         .mode("append")
         .saveAsTable("main.analytics.cdf_events"))

        print(f"  Captured {count} CDF events from {source_table} ✓")
        return count

    except Exception as e:
        print(f"  Error reading CDF from {source_table}: {e}")
        return 0

# Read CDF from each Silver table
total = 0
total += read_cdf_and_append(
    "main.silver.companies",
    key_col       = "ticker",
    snapshot_cols = ["ticker", "name", "exchange_name", "market_cap_billions"]
)
total += read_cdf_and_append(
    "main.silver.price_snapshots",
    key_col       = "ticker",
    snapshot_cols = ["ticker", "snapshot_date", "open", "close", "daily_return_pct"]
)
total += read_cdf_and_append(
    "main.silver.news_articles",
    key_col       = "article_id",
    snapshot_cols = ["ticker", "title", "sentiment", "article_age_days"]
)

print(f"\nTotal CDF events captured: {total}")


In [ ]:
# 4. Analytics on CDF events
print("\n--- CDF Analytics ---")

cdf = spark.table("main.analytics.cdf_events")

print(f"Total events in table: {cdf.count()}")

print("\nEvents by source table + operation:")
cdf.groupBy("source_table", "operation") \
   .count() \
   .orderBy("source_table", "operation") \
   .show(truncate=False)

print("\nEvents by ticker:")
cdf.filter(F.col("ticker").isNotNull()) \
   .groupBy("ticker") \
   .count() \
   .orderBy(F.col("count").desc()) \
   .show(10)

print("\nMost recent events:")
cdf.select("source_table", "operation", "ticker", "commit_version", "commit_ts") \
   .orderBy(F.col("commit_ts").desc()) \
   .show(10, truncate=False)


In [ ]:
# 5. Create CDF Summary Gold table
# Aggregated view of all pipeline changes — useful for monitoring and dashboards
print("\n--- Building CDF summary table ---")

cdf_summary = (
    spark.table("main.analytics.cdf_events")
    .groupBy("source_table", "operation")
    .agg(
        F.count("event_id").alias("event_count"),
        F.countDistinct("ticker").alias("distinct_tickers"),
        F.min("commit_ts").alias("first_event_ts"),
        F.max("commit_ts").alias("last_event_ts"),
        F.max("commit_version").alias("latest_version")
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT).cast("timestamp"))
    .orderBy("source_table", "operation")
)

(cdf_summary
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.analytics.cdf_summary"))

print("Written → main.analytics.cdf_summary ✓")
cdf_summary.show(truncate=False)


In [ ]:
# 6. Summary
print("\n=== Phase 6 CDF Summary ===")
print(f"Processed at: {PROCESSED_AT}\n")

for table in ["cdf_events", "cdf_summary"]:
    try:
        count = spark.table(f"main.analytics.{table}").count()
        print(f"  main.analytics.{table:<20} rows: {count:>6}")
    except Exception as e:
        print(f"  main.analytics.{table:<20} ERROR: {e}")

print("""
CDF Architecture:
  Silver Delta tables  (CDF enabled via enableChangeDataFeed)
         ↓ readChangeFeed = true (captures INSERT/UPDATE/DELETE)
  main.analytics.cdf_events   (every row-level change)
         ↓ groupBy aggregation
  main.analytics.cdf_summary  (pipeline monitoring view)

Lakebase CDF (equivalent):
  Lakebase tables  (CDF via REPLICA IDENTITY FULL)
         ↓ logical replication stream
  Delta analytics table  (same pattern, Postgres source)
  Note: Requires OAuth federation (not available on Free Edition)
""")
print("Phase 6 complete ✓")
